# {{ notebook_title }}

Generated perturbation probability-artifact analysis notebook for `{{ experiment_name }}`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

sns.set_theme(style="darkgrid", context="notebook")

ROOT = Path.cwd().parent.parent
if not (ROOT / "runs").exists():
    ROOT = ROOT.parent

SRC_PATH = ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from distill_ood_detection.evaluation.ood_metrics import ood_detection_metrics
from distill_ood_detection.evaluation.ood_scores import (
    absolute_max_probability_difference,
    logit_l2_distance,
    max_probability_difference,
    student_teacher_kl_divergence,
)

EXPERIMENT_NAME = "{{ experiment_name }}"
RUN_DIR = ROOT / "{{ run_dir }}"
PERTURBATION_LABEL = "Constant clipping"
PROBABILITY_DIR = RUN_DIR / "probabilities"
MANIFEST_PATH = PROBABILITY_DIR / "manifest.json"
CHECKPOINT = "best"

METHODS = {
    "mse_logits": "MSE Logits",
    "kl_divergence": "KL Divergence",
}
DATASETS = {
    "cifar10_test": "ID: CIFAR-10 test",
    "mnist_test": "Far-OOD: MNIST test",
    "svhn_test": "Far-OOD: SVHN test",
    "cifar100_test": "Near-OOD: CIFAR-100 test",
}
OOD_DATASET_LABELS = {
    "mnist_test": "Far-OOD: MNIST",
    "svhn_test": "Far-OOD: SVHN",
    "cifar100_test": "Near-OOD: CIFAR-100",
}
DISTRIBUTION_ORDER = ["ID", "Far-OOD", "Near-OOD"]
TEACHER_SCORE_NAME = "Teacher MSP"
STUDENT_MODEL_NAME = "Perturbation linear"
OOD_SCORE_FUNCTIONS = {
    "Max probability difference": max_probability_difference,
    "Absolute max probability difference": absolute_max_probability_difference,
    "Student-teacher KL divergence": student_teacher_kl_divergence,
}
LOGIT_L2_SCORE_NAME = "Centered logit L2 distance"
SIGNED = True

MANIFEST_PATH


In [ ]:
with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

manifest["experiment_name"], manifest["checkpoint_selection"]


In [ ]:
def load_probability_artifact(path: str | Path) -> dict[str, object]:
    """Load one saved probability artifact on CPU."""

    artifact_path = Path(path)
    if not artifact_path.is_absolute():
        artifact_path = ROOT / artifact_path
    return torch.load(artifact_path, map_location="cpu", weights_only=False)


artifacts = []
for entry in manifest["artifacts"]:
    artifact = load_probability_artifact(entry["path"])
    artifacts.append({**artifact, **entry})

len(artifacts)


In [ ]:
teacher_outputs = {
    artifact["dataset"]: artifact
    for artifact in artifacts
    if artifact["model"] == "teacher"
}
student_outputs = {
    (artifact["dataset"], artifact["method"], artifact["checkpoint"]): artifact
    for artifact in artifacts
    if artifact["model"] == "student"
}

METHOD_ORDER = sorted(
    {
        artifact["method"]
        for artifact in artifacts
        if artifact["model"] == "student"
        and artifact["checkpoint"] == CHECKPOINT
        and artifact["method"] in METHODS
    },
    key=list(METHODS).index,
)
DATASET_ORDER = [
    dataset_key
    for dataset_key in DATASETS
    if dataset_key in teacher_outputs
    and any(
        (dataset_key, method_key, CHECKPOINT) in student_outputs
        for method_key in METHOD_ORDER
    )
]

if not METHOD_ORDER:
    raise ValueError(
        f"No student probability artifacts found for checkpoint={CHECKPOINT!r} in {MANIFEST_PATH}."
    )


def to_numpy(tensor: torch.Tensor) -> np.ndarray:
    """Move a tensor to CPU and convert it to a NumPy array."""

    return tensor.detach().cpu().numpy()


def average_draw_outputs(outputs: np.ndarray) -> np.ndarray:
    """Average stochastic perturbation draws when outputs are three-dimensional."""

    if outputs.ndim == 3:
        return outputs.mean(axis=1)
    return outputs


def average_max_probability(probabilities: np.ndarray) -> np.ndarray:
    """Compute per-sample MSP, averaging over stochastic draws when present."""

    max_probabilities = probabilities.max(axis=-1)
    if max_probabilities.ndim == 2:
        return max_probabilities.mean(axis=1)
    return max_probabilities


def dataset_group(dataset_key: str) -> str:
    """Map each dataset to the ID/far-OOD/near-OOD grouping used in plots."""

    if dataset_key == "cifar10_test":
        return "ID"
    if dataset_key == "cifar100_test":
        return "Near-OOD"
    return "Far-OOD"


def build_argmax_heatmap(
    teacher_probs: np.ndarray,
    student_probs: np.ndarray,
    num_classes: int,
) -> np.ndarray:
    """Compute row-normalized teacher-vs-student argmax percentages."""

    teacher_argmax = average_draw_outputs(teacher_probs).argmax(axis=1)
    student_argmax = average_draw_outputs(student_probs).argmax(axis=1)
    counts = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(counts, (teacher_argmax, student_argmax), 1)
    row_totals = counts.sum(axis=1, keepdims=True)
    return np.divide(
        counts * 100.0,
        row_totals,
        out=np.zeros_like(counts, dtype=float),
        where=row_totals > 0,
    )


def build_comparison(dataset_key: str, method_key: str) -> dict[str, object]:
    """Assemble teacher/student comparison data for one dataset and method."""

    teacher = teacher_outputs[dataset_key]
    student = student_outputs[(dataset_key, method_key, CHECKPOINT)]
    teacher_logits = to_numpy(teacher["logits"])
    student_logits = to_numpy(student["logits"])
    teacher_probs = to_numpy(teacher["probabilities"])
    student_probs = to_numpy(student["probabilities"])

    return {
        "teacher_logits": teacher_logits,
        "student_logits": student_logits,
        "teacher_probs": teacher_probs,
        "student_probs": student_probs,
        "teacher_msp": average_max_probability(teacher_probs),
        "student_msp": average_max_probability(student_probs),
        "max_probability_difference": max_probability_difference(teacher_probs, student_probs),
        "absolute_max_probability_difference": absolute_max_probability_difference(
            teacher_probs,
            student_probs,
        ),
        "student_teacher_kl_divergence": student_teacher_kl_divergence(
            teacher_probs,
            student_probs,
        ),
        "logit_l2_distance": logit_l2_distance(teacher_logits, student_logits),
        "argmax_agreement": build_argmax_heatmap(
            teacher_probs,
            student_probs,
            num_classes=teacher_probs.shape[-1],
        ),
    }


comparisons = {
    (method_key, dataset_key): build_comparison(dataset_key, method_key)
    for method_key in METHOD_ORDER
    for dataset_key in DATASET_ORDER
    if (dataset_key, method_key, CHECKPOINT) in student_outputs
}


def build_grouped_metric_frame(metric_key: str, value_name: str) -> pd.DataFrame:
    """Collect one comparison metric into a dataframe by method and group."""

    frames = []
    for (method_key, dataset_key), comparison in comparisons.items():
        frames.append(
            pd.DataFrame(
                {
                    value_name: comparison[metric_key],
                    "distribution": dataset_group(dataset_key),
                    "dataset": DATASETS[dataset_key],
                    "method": METHODS[method_key],
                }
            )
        )
    return pd.concat(frames, ignore_index=True)


teacher_msp_frame = build_grouped_metric_frame("teacher_msp", "teacher_msp")
student_msp_frame = build_grouped_metric_frame("student_msp", "student_msp")
max_probability_difference_frame = build_grouped_metric_frame(
    "max_probability_difference",
    "max_probability_difference",
)
absolute_max_probability_difference_frame = build_grouped_metric_frame(
    "absolute_max_probability_difference",
    "absolute_max_probability_difference",
)
student_teacher_kl_divergence_frame = build_grouped_metric_frame(
    "student_teacher_kl_divergence",
    "student_teacher_kl_divergence",
)
logit_l2_distance_frame = build_grouped_metric_frame(
    "logit_l2_distance",
    "logit_l2_distance",
)

{
    "available_methods": [METHODS[method_key] for method_key in METHOD_ORDER],
    "available_datasets": [DATASETS[dataset_key] for dataset_key in DATASET_ORDER],
    "draws": next(iter(comparisons.values()))["teacher_probs"].shape[1],
    "comparison_shapes": {
        (method_key, dataset_key): {
            "teacher_probs_shape": comparison["teacher_probs"].shape,
            "student_probs_shape": comparison["student_probs"].shape,
        }
        for (method_key, dataset_key), comparison in comparisons.items()
    },
}


The histograms below compare teacher and student maximum probabilities by dataset group. For perturbation artifacts, each sample value is averaged over stochastic draws.


In [ ]:
def make_axes_grid(figsize: tuple[int, int] | None = None) -> tuple[plt.Figure, np.ndarray]:
    """Create a grid with one subplot per available method."""

    method_count = len(METHOD_ORDER)
    ncols = method_count
    nrows = int(np.ceil(method_count / ncols))
    if figsize is None:
        figsize = (7 * ncols, 5 * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[method_count:]:
        ax.set_visible(False)
    return fig, axes


deep_palette = sns.color_palette("deep")
dark_palette = sns.color_palette("dark")
distribution_palette = {
    "ID": deep_palette[0],
    "Far-OOD": deep_palette[3],
    "Near-OOD": dark_palette[2],
}


def plot_grouped_histograms(
    frame: pd.DataFrame,
    value_name: str,
    xlabel: str,
    title: str,
    *,
    bins: int | np.ndarray = 40,
    kde: bool = True,
) -> None:
    """Plot grouped histograms for each available method."""

    fig, axes = make_axes_grid()
    for axis_index, method_key in enumerate(METHOD_ORDER):
        ax = axes[axis_index]
        method_name = METHODS[method_key]
        method_frame = frame.loc[frame["method"] == method_name]
        sns.histplot(
            data=method_frame,
            x=value_name,
            hue="distribution",
            hue_order=DISTRIBUTION_ORDER,
            palette=distribution_palette,
            bins=bins,
            kde=kde,
            stat="percent",
            common_norm=False,
            ax=ax,
        )
        ax.set_title(method_name)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Percent")
        legend = ax.get_legend()
        if legend is not None:
            legend.set_title("Dataset group")
    fig.suptitle(title, fontsize=13)


In [ ]:
probability_bins = np.linspace(0.0, 1.0, 41)

plot_grouped_histograms(
    teacher_msp_frame,
    value_name="teacher_msp",
    xlabel="Teacher maximum probability averaged over perturbation draws",
    title=f"{PERTURBATION_LABEL}: Teacher MSP Histograms",
    bins=probability_bins,
    kde=False,
)
plot_grouped_histograms(
    student_msp_frame,
    value_name="student_msp",
    xlabel="Student maximum probability averaged over perturbation draws",
    title=f"{PERTURBATION_LABEL}: Student MSP Histograms",
    bins=probability_bins,
    kde=False,
)


The next histograms show teacher-student OOD score distributions. Lower signed scores are more OOD-like by project convention, while raw distances/divergences are shown here for interpretability.


In [ ]:
plot_grouped_histograms(
    max_probability_difference_frame,
    value_name="max_probability_difference",
    xlabel="E[max_proba_teacher - max_proba_student]",
    title=f"{PERTURBATION_LABEL}: Maximum Probability Difference Histograms",
)
plot_grouped_histograms(
    absolute_max_probability_difference_frame,
    value_name="absolute_max_probability_difference",
    xlabel="E[abs(max_proba_teacher - max_proba_student)]",
    title=f"{PERTURBATION_LABEL}: Absolute Maximum Probability Difference Histograms",
)
plot_grouped_histograms(
    student_teacher_kl_divergence_frame,
    value_name="student_teacher_kl_divergence",
    xlabel="E[KL(teacher || student)]",
    title=f"{PERTURBATION_LABEL}: Student-Teacher KL Divergence Histograms",
)
plot_grouped_histograms(
    logit_l2_distance_frame,
    value_name="logit_l2_distance",
    xlabel="E[L2 distance between centered teacher and student logits]",
    title=f"{PERTURBATION_LABEL}: Centered Logit L2 Distance Histograms",
)


In [ ]:
fig, axes = plt.subplots(
    len(METHOD_ORDER),
    len(DISTRIBUTION_ORDER),
    figsize=(21, 4.5 * len(METHOD_ORDER)),
    constrained_layout=True,
    squeeze=False,
)
distribution_cmaps = {
    distribution: sns.light_palette(distribution_palette[distribution], as_cmap=True)
    for distribution in DISTRIBUTION_ORDER
}

for row_index, method_key in enumerate(METHOD_ORDER):
    for col_index, distribution in enumerate(DISTRIBUTION_ORDER):
        ax = axes[row_index, col_index]
        heatmaps = [
            comparison["argmax_agreement"]
            for (candidate_method_key, dataset_key), comparison in comparisons.items()
            if candidate_method_key == method_key and dataset_group(dataset_key) == distribution
        ]
        if not heatmaps:
            ax.set_visible(False)
            continue
        heatmap = np.mean(heatmaps, axis=0)
        sns.heatmap(
            heatmap,
            ax=ax,
            cmap=distribution_cmaps[distribution],
            vmin=0,
            vmax=100,
            annot=True,
            fmt=".1f",
            cbar=True,
        )
        ax.set_title(f"{METHODS[method_key]} | {distribution}")
        ax.set_xlabel("Student argmax class")
        ax.set_ylabel("Teacher argmax class")

fig.suptitle(
    f"{PERTURBATION_LABEL}: Teacher vs Student Argmax Agreement (%)",
    fontsize=13,
)


## ROC-AUC metrics

Each table cell reports `ROC-AUC/FPR@95TPR`. Higher OOD scores are treated as more ID-like.


In [ ]:
def teacher_msp_scores(artifact: dict[str, object]) -> np.ndarray:
    """Compute teacher MSP scores, averaging over perturbation draws."""

    return average_max_probability(to_numpy(artifact["probabilities"]))


def build_ood_metric_table() -> pd.DataFrame:
    """Build a ROC-AUC/FPR@95 summary table for each OOD score variant."""

    id_dataset_key = "cifar10_test"
    ood_dataset_keys = [
        dataset_key for dataset_key in OOD_DATASET_LABELS if dataset_key in DATASET_ORDER
    ]
    column_index = pd.MultiIndex.from_tuples(
        [("CIFAR-10", OOD_DATASET_LABELS[ood_dataset_key]) for ood_dataset_key in ood_dataset_keys],
        names=["ID dataset", "OOD dataset"],
    )
    row_index = pd.MultiIndex.from_tuples(
        [("Teacher", "-", TEACHER_SCORE_NAME)]
        + [
            (STUDENT_MODEL_NAME, METHODS[method_key], score_name)
            for method_key in METHOD_ORDER
            for score_name in [*OOD_SCORE_FUNCTIONS, LOGIT_L2_SCORE_NAME]
        ],
        names=["Model", "Distillation method", "OOD score"],
    )
    metric_rows: dict[tuple[str, str, str], dict[tuple[str, str], str]] = {
        row_key: {} for row_key in row_index
    }

    id_teacher = teacher_outputs[id_dataset_key]
    id_teacher_logits = to_numpy(id_teacher["logits"])
    id_teacher_probs = to_numpy(id_teacher["probabilities"])
    id_teacher_msp_scores = teacher_msp_scores(id_teacher)

    for ood_dataset_key in ood_dataset_keys:
        ood_teacher = teacher_outputs[ood_dataset_key]
        ood_teacher_logits = to_numpy(ood_teacher["logits"])
        ood_teacher_probs = to_numpy(ood_teacher["probabilities"])
        ood_teacher_msp_scores = teacher_msp_scores(ood_teacher)
        labels = np.concatenate(
            [
                np.ones(id_teacher_msp_scores.shape[0], dtype=int),
                np.zeros(ood_teacher_msp_scores.shape[0], dtype=int),
            ]
        )
        column_key = ("CIFAR-10", OOD_DATASET_LABELS[ood_dataset_key])

        teacher_metrics = ood_detection_metrics(
            labels,
            np.concatenate([id_teacher_msp_scores, ood_teacher_msp_scores]),
        )
        metric_rows[("Teacher", "-", TEACHER_SCORE_NAME)][column_key] = (
            f"{teacher_metrics['roc_auc']:.2f}/{teacher_metrics['fpr_at_95_tpr']:.2f}"
        )

        for method_key in METHOD_ORDER:
            id_student = student_outputs[(id_dataset_key, method_key, CHECKPOINT)]
            ood_student = student_outputs[(ood_dataset_key, method_key, CHECKPOINT)]
            id_student_logits = to_numpy(id_student["logits"])
            ood_student_logits = to_numpy(ood_student["logits"])
            id_student_probs = to_numpy(id_student["probabilities"])
            ood_student_probs = to_numpy(ood_student["probabilities"])

            for score_name, score_function in OOD_SCORE_FUNCTIONS.items():
                id_scores = score_function(id_teacher_probs, id_student_probs, signed=SIGNED)
                ood_scores = score_function(ood_teacher_probs, ood_student_probs, signed=SIGNED)
                metrics = ood_detection_metrics(labels, np.concatenate([id_scores, ood_scores]))
                metric_rows[(STUDENT_MODEL_NAME, METHODS[method_key], score_name)][column_key] = (
                    f"{metrics['roc_auc']:.2f}/{metrics['fpr_at_95_tpr']:.2f}"
                )

            id_scores = logit_l2_distance(id_teacher_logits, id_student_logits, signed=SIGNED)
            ood_scores = logit_l2_distance(ood_teacher_logits, ood_student_logits, signed=SIGNED)
            metrics = ood_detection_metrics(labels, np.concatenate([id_scores, ood_scores]))
            metric_rows[(STUDENT_MODEL_NAME, METHODS[method_key], LOGIT_L2_SCORE_NAME)][column_key] = (
                f"{metrics['roc_auc']:.2f}/{metrics['fpr_at_95_tpr']:.2f}"
            )

    metric_table = pd.DataFrame.from_dict(metric_rows, orient="index", columns=column_index)
    metric_table.index = row_index
    return metric_table


roc_auc_metric_table = build_ood_metric_table()
roc_auc_metric_table


In [ ]:
from sklearn.metrics import roc_curve


def build_single_ood_roc_curve(
    labels: np.ndarray,
    scores: np.ndarray,
    *,
    ood_dataset_label: str,
) -> dict[str, object]:
    """Build one ROC curve and its FPR@95TPR marker point."""

    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    metrics = ood_detection_metrics(labels, scores)
    candidate_indices = np.flatnonzero(tpr >= 0.95)
    marker_index = candidate_indices[np.argmin(fpr[candidate_indices])]
    return {
        "fpr": fpr,
        "tpr": tpr,
        "fpr_at_95_tpr": metrics["fpr_at_95_tpr"],
        "tpr_at_95_marker": float(tpr[marker_index]),
        "roc_auc": metrics["roc_auc"],
        "ood_dataset_label": ood_dataset_label,
        "legend_label": (
            f"{ood_dataset_label} "
            f"({metrics['roc_auc']:.4f}/{metrics['fpr_at_95_tpr']:.4f})"
        ),
    }


def build_student_ood_roc_curves() -> dict[tuple[str, str], list[dict[str, object]]]:
    """Build ROC curve data grouped by student method and OOD score."""

    id_dataset_key = "cifar10_test"
    score_names = [TEACHER_SCORE_NAME, *OOD_SCORE_FUNCTIONS, LOGIT_L2_SCORE_NAME]
    curves_by_axis = {
        (method_key, score_name): []
        for method_key in METHOD_ORDER
        for score_name in score_names
    }
    id_teacher = teacher_outputs[id_dataset_key]
    id_teacher_logits = to_numpy(id_teacher["logits"])
    id_teacher_probs = to_numpy(id_teacher["probabilities"])
    id_teacher_msp_scores = teacher_msp_scores(id_teacher)

    for ood_dataset_key, ood_dataset_label in OOD_DATASET_LABELS.items():
        if ood_dataset_key not in DATASET_ORDER:
            continue
        ood_teacher = teacher_outputs[ood_dataset_key]
        ood_teacher_logits = to_numpy(ood_teacher["logits"])
        ood_teacher_probs = to_numpy(ood_teacher["probabilities"])
        ood_teacher_msp_scores = teacher_msp_scores(ood_teacher)
        labels = np.concatenate(
            [
                np.ones(id_teacher_msp_scores.shape[0], dtype=int),
                np.zeros(ood_teacher_msp_scores.shape[0], dtype=int),
            ]
        )

        for method_key in METHOD_ORDER:
            curves_by_axis[(method_key, TEACHER_SCORE_NAME)].append(
                build_single_ood_roc_curve(
                    labels,
                    np.concatenate([id_teacher_msp_scores, ood_teacher_msp_scores]),
                    ood_dataset_label=ood_dataset_label,
                )
            )
            id_student = student_outputs[(id_dataset_key, method_key, CHECKPOINT)]
            ood_student = student_outputs[(ood_dataset_key, method_key, CHECKPOINT)]
            id_student_logits = to_numpy(id_student["logits"])
            ood_student_logits = to_numpy(ood_student["logits"])
            id_student_probs = to_numpy(id_student["probabilities"])
            ood_student_probs = to_numpy(ood_student["probabilities"])

            for score_name, score_function in OOD_SCORE_FUNCTIONS.items():
                id_scores = score_function(id_teacher_probs, id_student_probs, signed=SIGNED)
                ood_scores = score_function(ood_teacher_probs, ood_student_probs, signed=SIGNED)
                curves_by_axis[(method_key, score_name)].append(
                    build_single_ood_roc_curve(
                        labels,
                        np.concatenate([id_scores, ood_scores]),
                        ood_dataset_label=ood_dataset_label,
                    )
                )

            id_scores = logit_l2_distance(id_teacher_logits, id_student_logits, signed=SIGNED)
            ood_scores = logit_l2_distance(ood_teacher_logits, ood_student_logits, signed=SIGNED)
            curves_by_axis[(method_key, LOGIT_L2_SCORE_NAME)].append(
                build_single_ood_roc_curve(
                    labels,
                    np.concatenate([id_scores, ood_scores]),
                    ood_dataset_label=ood_dataset_label,
                )
            )

    return curves_by_axis


roc_curves_by_axis = build_student_ood_roc_curves()
score_names = [TEACHER_SCORE_NAME, *OOD_SCORE_FUNCTIONS, LOGIT_L2_SCORE_NAME]
fig, axes = plt.subplots(
    len(METHOD_ORDER),
    len(score_names),
    figsize=(7 * len(score_names), 4.8 * len(METHOD_ORDER)),
    constrained_layout=True,
    squeeze=False,
)
ood_palette = dict(
    zip(
        OOD_DATASET_LABELS.values(),
        sns.color_palette("deep", n_colors=len(OOD_DATASET_LABELS)),
        strict=True,
    )
)

for row_index, method_key in enumerate(METHOD_ORDER):
    for col_index, score_name in enumerate(score_names):
        ax = axes[row_index, col_index]
        curves = roc_curves_by_axis[(method_key, score_name)]
        for curve in curves:
            color = ood_palette[curve["ood_dataset_label"]]
            ax.plot(
                curve["fpr"],
                curve["tpr"],
                color=color,
                linewidth=1.8,
                alpha=0.85,
                label=curve["legend_label"],
            )
            ax.scatter(
                curve["fpr_at_95_tpr"],
                curve["tpr_at_95_marker"],
                color=color,
                edgecolor="black",
                linewidth=0.5,
                s=38,
                zorder=3,
            )
        ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1, alpha=0.45)
        ax.axhline(0.95, color="black", linestyle=":", linewidth=1, alpha=0.7)
        ax.set_title(f"{METHODS[method_key]} | {score_name}")
        ax.set_xlabel("False positive rate")
        ax.set_ylabel("True positive rate")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.01)
        ax.legend(title="OOD dataset (ROC-AUC/FPR@95)", fontsize=10, title_fontsize=10)

fig.suptitle(f"{PERTURBATION_LABEL}: Student OOD ROC Curves", fontsize=13)
